# 1. Entendendo e limpando os dados do 156

Para responder perguntas de gestão (onde está a demanda, quanto tempo cada serviço leva, se alguma região espera mais, quanto da fila está parada) os dados precisam de duas garantias: cada chamado contado uma vez, e com a situação e as datas corretas. Este notebook abre os arquivos brutos do portal, registra o que foi encontrado e a decisão tomada em cada caso.

As regras ficam em [`src/transform/cleaning.py`](../src/transform/cleaning.py), o mesmo código usado pelo pipeline e coberto pelos testes. Os números se referem ao recorte padrão, 2024 e 2025.

In [1]:
import pandas as pd

from src.config import YEARS
from src.ingestion.ckan import ingest
from src.transform.cleaning import (
    TICKET_KEY,
    clean_coordinates,
    clean_tickets,
    combine_snapshots,
    deduplicate_snapshots,
    normalize_text,
    read_raw_csv,
)

latest, previous = YEARS[-1], YEARS[-2]
raw = {year: read_raw_csv(path) for year, path in zip(YEARS, ingest(YEARS), strict=True)}
pd.Series({year: len(df) for year, df in raw.items()}, name="linhas")

2024    108082
2025    114653
Name: linhas, dtype: int64

## Primeiro olhar

In [2]:
raw[previous].head(3)

,GRUPOSERVICO_CODIGO,GRUPOSERVICO_DESCRICAO,SERVICO_CODIGO,SERVICO_DESCRICAO,LOGRADOURO,NUMERO,BAIRRO,RPA,DATA_DEMANDA,SITUACAO,DATA_ULT_SITUACAO,latitude,longitude
0,2,LIMPEZA URBANA,2,REMOCAO DE ANIMAIS MORTOS PEQUENOS,1TR da Campinas,65,Jardim Sao Paulo,5,2024-08-03,ATENDIDA,2024-08-05,-8.0852089,-34.9490991
1,2,LIMPEZA URBANA,2,REMOCAO DE ANIMAIS MORTOS PEQUENOS,2TR So Nos Dois,8,Passarinho,3,2024-08-12,ATENDIDA,2024-08-14,-7.989593850833942,-34.92590954314152
2,2,LIMPEZA URBANA,2,REMOCAO DE ANIMAIS MORTOS PEQUENOS,3TR Sucupira,1459,Iputinga,4,2024-04-12,ATENDIDA,2024-04-12,-8.028293792543325,-34.9343494533275


Cada linha é um chamado com 13 colunas, todas chegando como texto. Duas coisas aparecem logo: não existe número de protocolo que identifique o chamado, e os campos vêm com espaços de preenchimento (o `NUMERO` tem largura fixa).

## Descoberta 1: o arquivo de um ano não traz só chamados daquele ano

In [3]:
opened_year = pd.to_datetime(raw[latest]["DATA_DEMANDA"]).dt.year
opened_year.value_counts().sort_index(ascending=False).head(6)

DATA_DEMANDA
2025    105093
2024      4626
2023       933
2022       853
2021       984
2020       490
Name: count, dtype: int64

O arquivo de 2025 tem 9.560 chamados abertos em anos anteriores, alguns desde 2010. Ele funciona como uma foto do fim do ano: traz todo chamado que mudou de situação em 2025, não importa quando foi aberto. Contar as linhas de cada arquivo como "chamados do ano" inflaria a demanda e misturaria anos.

Decisão: o ano de um chamado é o da abertura (`DATA_DEMANDA`), e só entram chamados abertos dentro do recorte.

## Descoberta 2: o mesmo chamado aparece em mais de um arquivo, com situações diferentes

Sem protocolo, a forma de reconhecer o mesmo chamado em arquivos diferentes é a combinação de serviço, endereço, bairro e data de abertura.

In [4]:
clean = {year: clean_tickets(df, year) for year, df in raw.items()}
repeated_share = clean[latest].duplicated(TICKET_KEY).mean()
print(f"Linhas que repetem a combinação dentro do arquivo de {latest}: {repeated_share:.1%}")

in_both = clean[latest].merge(clean[previous], on=TICKET_KEY, suffixes=("_now", "_before"))
pd.crosstab(in_both["status_before"], in_both["status_now"], margins=True, margins_name="total")

Linhas que repetem a combinação dentro do arquivo de 2025: 2.6%


status_now,ATENDIDA,CADASTRADA,EXECUCAO,PENDENCIA,PREPARACAO,total
status_before,,,,,,
ATENDIDA,35,0,0,0,0,35
CADASTRADA,666,7,13,0,126,812
EXECUCAO,1575,0,1,37,0,1613
FISCALIZACAO,20,0,0,0,0,20
PENDENCIA,110,0,3,6,0,119
PREPARACAO,420,0,9,8,2,439
total,2826,7,26,51,128,3038


Cerca de 3 mil chamados do arquivo de 2025 já estavam no de 2024, e quase todos mudaram de situação no caminho, a maioria para `ATENDIDA`. Empilhar os arquivos contaria esses chamados duas vezes, uma delas com a situação velha, distorcendo o volume e a taxa de atendimento.

Decisão: manter só a versão mais recente de cada chamado (`deduplicate_snapshots`). As linhas que repetem a combinação dentro de um mesmo arquivo são tratadas como chamados distintos (dois pedidos iguais no mesmo dia e endereço), uma aproximação necessária sem protocolo.

## Descoberta 3: grafias diferentes e valores que querem dizer "não informado"

In [5]:
neighborhood = raw[previous]["BAIRRO"]
neighborhood[neighborhood.str.contains("  |INFORMADO|INDEFINIDO")].value_counts().head(5)

BAIRRO
Cohab   Ibura de Cima           3500
Alto do Mandu   Sitio Grande     295
Sitio dos Pintos   Sao Bras      263
BAIRRO NAO INFORMADO              54
Name: count, dtype: int64

In [6]:
raw[previous]["NUMERO"].str.strip().value_counts().head(4)

NUMERO
SN     17978
S/N    10112
00      9906
0       1627
Name: count, dtype: int64

Há bairros com espaços repetidos (`Cohab   Ibura de Cima`, `Alto do Mandu   Sitio Grande`) e o placeholder `BAIRRO NAO INFORMADO`, que entraria nos rankings como se fosse um bairro. No número do imóvel, os quatro valores mais comuns (`SN`, `S/N`, `00` e `0`) querem dizer a mesma coisa: endereço sem número.

Decisão: textos em maiúsculas, sem acento e com espaços únicos, para que o mesmo bairro sempre se agrupe junto; placeholders de bairro e de número viram nulo.

In [7]:
normalize_text(pd.Series(["Alto do Mandu   Sitio Grande", "Jardim São Paulo"])).tolist()

['ALTO DO MANDU SITIO GRANDE', 'JARDIM SAO PAULO']

## Descoberta 4: coordenadas incompletas e algumas fora do mapa

In [8]:
coordinates = {}
for year, df in raw.items():
    latitude = df["latitude"].str.strip().replace("", None)
    valid_latitude, _ = clean_coordinates(latitude, df["longitude"].str.strip())
    coordinates[year] = {
        "% com coordenadas": round(100 * latitude.notna().mean(), 1),
        "fora do Recife": int((latitude.notna() & valid_latitude.isna()).sum()),
    }
pd.DataFrame.from_dict(coordinates, orient="index")

,% com coordenadas,fora do Recife
2024,23.0,10
2025,49.8,47


Só 23% das linhas de 2024 e metade das de 2025 têm coordenadas, e algumas caem fora do Recife (latitude 85, longitude -180, típicas de erro de geocodificação). Um mapa de pontos ficaria enviesado para os chamados mais recentes.

Decisão: coordenadas fora da cidade viram nulo, e as comparações regionais usam bairro e RPA, que vêm sempre preenchidos.

## Descoberta 5: por que o recorte começa em 2024

In [9]:
raw_2023 = read_raw_csv(ingest([2023])[0])


# Group name behind each group code, so the same code can be compared across years
def group_names(df: pd.DataFrame) -> pd.Series:
    codes = df["GRUPOSERVICO_CODIGO"].str.strip()
    return df.groupby(codes)["GRUPOSERVICO_DESCRICAO"].first().str.strip()


pd.DataFrame({"2023": group_names(raw_2023), "2024": group_names(raw[2024])}).dropna()

,2023,2024
GRUPOSERVICO_CODIGO,,
11,DRENAGEM,FISCALIZACAO URBANA
12,ESCADARIA,ILUMINACAO PROVISORIA
15,FISCALIZAO URBANA,LUMINARIAS
16,ILUMINAO PROVISRIA,MACRODRENAGEM
17,ILUMINAO PBLICA,MANUT. URBANA
20,LIMPEZA URBANA,COLETA URBANA
23,MANUT. URBANA,ACAO EDUCATIVA
24,NECROPOLES,PROJETOS
25,PAVIMENTAO,PRACAS


In [10]:
raw_2023["SITUACAO"].str.strip().unique()

<StringArray>
['CADASTRADA', 'ATENDIDA', 'PREPARAO', 'EXECUO', 'FISCALIZAO', 'PENDENTE']
Length: 6, dtype: str

Até 2023 os mesmos códigos significam outra coisa: o 17 era iluminação pública e passou a ser manutenção urbana, o 20 era limpeza urbana e virou coleta urbana. Os acentos também foram cortados na origem (`PREPARAO`, `EXECUO`), e `PENDENTE` virou `PENDENCIA`. Juntar esses anos exigiria um de-para manual de mais de cem serviços, com risco real de atribuir chamados ao serviço errado.

Decisão: o recorte começa em 2024, e o pipeline recusa anos anteriores explicando o motivo.

## Resultado: dos arquivos brutos aos chamados analisados

In [11]:
stacked = pd.concat(clean.values(), ignore_index=True)
tickets = combine_snapshots(list(clean.values()), YEARS)
pd.Series(
    {
        "linhas nos arquivos": sum(len(df) for df in raw.values()),
        "uma versão por chamado": len(deduplicate_snapshots(stacked)),
        f"abertos entre {YEARS[0]} e {YEARS[-1]}": len(tickets),
    },
    name="linhas",
)

linhas nos arquivos          222735
uma versão por chamado       219802
abertos entre 2024 e 2025    204408
Name: linhas, dtype: int64

In [12]:
tickets["status"].value_counts()

status
ATENDIDA        157728
CADASTRADA       27858
PREPARACAO        9608
EXECUCAO          5661
PENDENCIA         3449
FISCALIZACAO       104
Name: count, dtype: int64

## O que levar para a análise

- 204.408 chamados abertos em 2024 e 2025, cada um contado uma vez e com a situação mais recente conhecida (fim de 2025).
- `ATENDIDA` é tratada como chamado resolvido; as demais situações (cadastrada, preparação, execução, pendência, fiscalização) formam a fila em aberto, 23% dos chamados.
- O tempo de resolução é a diferença entre a abertura e a última mudança de situação e só existe para chamados atendidos. Em serviços com fila grande essa mediana subestima a espera, então a fila em aberto precisa ser olhada junto.
- Comparações regionais usam bairro e RPA, não coordenadas.

O resultado é carregado no PostgreSQL pelo pipeline e analisado em [02_business_analysis.ipynb](02_business_analysis.ipynb).